## RAG 공식 문서
https://docs.langchain.com/oss/python/langchain/rag?_gl=1*1jm9b9x*_gcl_au*MTc2NzgyNTE2Mi4xNzY4MjkyMDkz*_ga*MTE4NzUxMTgwMy4xNzY4MjkyMDkz*_ga_47WX3HKKY2*czE3NjgzNzQxMzQkbzIkZzEkdDE3NjgzNzUyMDUkajYwJGwwJGgw

```text
1.문서읽기
2.문서나누기(chunk)
3.embedding
4.vector db 저장하기
5.유사도검색을 통해 문서추출
6.추출된문서(context:유사도검색을 통해 검색된 문서의합본)

context:

question:내 질문
answer:


In [28]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

template = "{topic}에 대해 {adjective} 농담 한마디 해줘."
prompt_template = ChatPromptTemplate.from_template(template) #HumanMessage로 return 함
prompt = prompt_template.invoke({"topic": "인공지능", "adjective": "썰렁한"})
print(prompt)


prompt_template = PromptTemplate.from_template(template)
prompt = prompt_template.invoke({"topic": "인공지능", "adjective": "썰렁한"})
print(prompt)
prompt = prompt_template.format(topic="인공지능", adjective="썰렁한")
print(prompt)

messages=[HumanMessage(content='인공지능에 대해 썰렁한 농담 한마디 해줘.', additional_kwargs={}, response_metadata={})]
text='인공지능에 대해 썰렁한 농담 한마디 해줘.'
인공지능에 대해 썰렁한 농담 한마디 해줘.


랭체인 (두 버전 차이가 多)

-------------

- 0.x 버전 : pip install langchain_classic
- 1.x 버전
- 1.0, 1.1 ~

In [29]:
import langchain
langchain.__version__

'1.2.3'

In [ ]:
%pip install -U langchain-text-splitters # LangChain의 텍스트 분할

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, JSONLoader, CSVLoader, WebBaseLoader, ImageCaptionLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [32]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1.문서읽기

In [ ]:
pdfloader = PyPDFLoader('pdf/The_Adventures_of_Tom_Sawyer.pdf')
document = pdfloader.load() # PDF 파일을 읽어서 LangChain의 Document 리스트(페이지별 텍스트+메타데이터)로 변환
print( document )

[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'pdf/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content=''), Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'pdf/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 1, 'page_label': '2'}, page_content='PENGUIN   READERS  2000  \n \n \n  \n \n  \n \nwww.penguinreaders.com'), Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9

In [61]:
document[2]

Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'pdf/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 2, 'page_label': '3'}, page_content='The Adventures of                 \nTom Sawyer \n \nMARK TWAIN \nLevel 1 \n \nRetold by Jacqueline Kehl                                                    \nSeries Editors: Andy Hopkins and Jocelyn Potter')

In [58]:
document[2].page_content

'The Adventures of                 \nTom Sawyer \n \nMARK TWAIN \nLevel 1 \n \nRetold by Jacqueline Kehl                                                    \nSeries Editors: Andy Hopkins and Jocelyn Potter'

In [59]:
document[2].metadata['page_label'] 

'3'

In [60]:
document[2].metadata['source']

'pdf/The_Adventures_of_Tom_Sawyer.pdf'

In [54]:
len( document)

35

In [55]:
document

[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'pdf/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'pdf/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 1, 'page_label': '2'}, page_content='PENGUIN   READERS  2000  \n \n \n  \n \n  \n \nwww.penguinreaders.com'),
 Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5

In [ ]:
for doc in document:
    print( doc.page_content) # 페이지 단위로 텍스트 출력
    print('----------------')


----------------
PENGUIN   READERS  2000  
 
 
  
 
  
 
www.penguinreaders.com
----------------
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter
----------------
Pearson Education Limited                                                                            
Edinburgh Gate, Harlow,                                                                               
Essex CM20 2JE, England                                                                              
and Associated Companies throughout the world. 
ISBN 0 582 41923 9 
 
First published 1876                                                                                  
Published by Puffin Books 1950                                                                         
This edition first published 2000 
Copyright © Penguin Books 2000 
Typeset by Digital Type, London 
Set

### 2.문서 자르기

In [37]:
txt_split = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
docs = txt_split.split_documents( document)

In [38]:
docs

[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'pdf/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 1, 'page_label': '2'}, page_content='PENGUIN   READERS  2000  \n \n \n  \n \n  \n \nwww.penguinreaders.com'),
 Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'pdf/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 2, 'page_label': '3'}, page_content='The Adventures of                 \nTom Sawyer \n \nMARK TWAIN \nLevel 1 \n

In [ ]:
len( docs )

79

### 3.embedding

In [40]:
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
embedding = OpenAIEmbeddings( model='text-embedding-ada-002')
## 벡터db에 저장되는 종류
## 1. page_content : embdding 저장
## 2. 원본텍스트도 저장
## 3. 각종정보:pdf이름 페이지번호,..
vdb = FAISS.from_documents( docs, embedding)

In [42]:
# vdb.save_local('myfaiss')
# FAISS.load_local('myfaiss')

### 4. 유사도 검색

In [ ]:
# 질문과 가장 비슷한 내용이 들어있는 문서 조각 5개를 벡터DB에서 검색해서, 그 텍스트를 출력
retrieved_docs = vdb.similarity_search("등장인물에 대해 알려줄래", k=5)
# print(retrieved_docs)
for doc in retrieved_docs:
    print( doc.page_content)
    print('-------------')

A man asked him, “Where were you on the night of  
June 17th?” 
“I was in the graveyard,” Tom answered. 
“Did you see any people there?” the man asked: 
“Yes. Injun Joe, the doctor, and Muff Potter were there. 
They didn’t see me because I was behind some big trees.” 
“What did you see?” the man asked. 
“Injun Joe and the doctor talked angrily,” Tom 
answered. “Then Injun Joe killed the doctor with his knife. 
Muff Potter didn’t do it.” 
The people at the trial were surprised. Injun Joe quickly
-------------
and went to their places. The teacher looked at his book. 
“Who did this? Who tore my book?” he asked angrily. 
The room was very quiet. The teacher started to ask 
every child, “Did you do this?” 
They answered, “No, I didn’t.” 
Then he looked at Becky “Becky, did you do this?” 
 
 
 
11
-------------
talking about it. Becky wanted to talk to Tom, but he 
didn’t look at her. 
Then Tom talked to Amy. Becky watched him and she 
was angry. She said to her friends, “I’m going to have 

```python

- vdb.similarity_search("등장인물에 대해 알려줄래", k=5)

#가장 유사한 문서 3개만 가져와라
- retriever = vdb.as_retriever(search_kwargs={"k": 3}) 

# 유사도 점수가 0.7 이상인 것만 가져와라
- retriever = vdb.as_retriever(search_type="similarity_score_threshold", 
                search_kwargs={"score_threshold": 0.7})

---

#### similarity_search vs vdb.as_retriever

**"지금 당장 결과를 받을 것인가(실행)"** 아니면 **"자동화된 공정의 부품으로 만들 것인가(설정)"** 의 차이.

---

### 1. `vdb.similarity_search(...)`: 즉시 실행형

이 방식은 코드를 실행하는 즉시 결과를 리스트로 반환.

* **성격:** 단순 함수 호출.
* **결과값:** `[Document, Document, ...]` (검색된 문서 객체 5개가 담긴 파이스트 리스트).
* **용도:** * 디버깅할 때 (내 데이터가 잘 저장됐나? 검색이 잘 되나?).
* 간단하게 검색 결과만 화면에 뿌려줄 때.


* **특징:** 질문("등장인물에 대해 알려줄래")을 **직접** 넣어줘야.

```python
# 실행하자마자 결과 5개가 변수 docs에 담김
docs = vdb.similarity_search("등장인물에 대해 알려줄래", k=5)

```

### 2. `vdb.as_retriever(...)`: 조립용 부품형 (추천)

이 방식은 결과를 바로 가져오는 게 아니라, 검색을 수행할 수 있는 **'검색기(Retriever) 객체'**를 만든다.

* **성격:** 설정(Configuration).
* **결과값:** `VectorStoreRetriever`라는 이름의 **객체(도구)**.
* **용도:** * LangChain의 체인(`|`)에 연결할 때.
* 사용자의 질문이 들어오면 자동으로 검색하도록 시스템을 설계할 때.

* **특징:** 질문을 미리 넣지 않는다. "앞으로 질문이 들어오면 k=3으로 검색해라"라는 **규칙**만 정해두는 것.

```python
# 검색 도구를 만드는 설정 과정. 아직 검색은 시작되지 않음.
retriever = vdb.as_retriever(search_kwargs={"k": 3})

# 나중에 체인에서 질문이 들어오면 이때 작동함
# result = retriever.invoke("질문 내용")

```

---

### 한눈에 비교하기

| 구분 | `similarity_search` | `as_retriever` |
| --- | --- | --- |
| **비유** | 마트에서 사과 5개를 **직접 골라 담기** | "사과를 3개씩 담아줘"라고 적힌 **로봇 설정하기** |
| **반환값** | 문서 리스트 (`List[Document]`) | 검색기 객체 (`Retriever`) |
| **체인 사용** | 불가능 (데이터일 뿐임) | **가능** (핵심 부품) |
| **k값 의미** | "지금 5개 가져와" | "앞으로 누가 물어보면 3개씩 찾아와" |

### 왜 RAG에서는 `as_retriever`를 쓰나

우리가 만든 체인(`qa = ( {"context": retriever | ... )`)은 사용자가 어떤 질문을 던질지 미리 알 수 없다. 그래서 **"질문이 들어오면 k=3으로 찾아와라"**라는 규칙을 가진 `retriever`를 부품으로 끼워 넣는 것.

만약 `similarity_search`를 체인 중간에 넣으려고 하면, 질문이 들어오기도 전에 검색을 시도하려 하기 때문에 에러가 발생하게 된다.


```text
문서:{context}
질문:{question}
답변:

In [46]:
# 6. 프롬프트 템플릿 정의
template = """질문에 대해 아래의 제공된 문맥(context)만을 사용하여 답변하세요. 
답을 모른다면 모른다고 말하고 답변을 지어내지 마세요. 
최대한 세 문장 내로 간결하게 답변하세요.

Question: {question} 
Context: {context} 
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI( model='gpt-4o-mini', temperature=0)
retriever = vdb.as_retriever() # vdb.as_retriever(k=4) 

def format_docs(docs):
    # print( "문서:",doc.page_content)
    return "\n\n".join( doc.page_content for doc in docs )

qa = ( 
        {"context": retriever | format_docs, "question": RunnablePassthrough()} #유사도 검색에서 반환된 문서들을 하나의 문자열로 변환
        | prompt  
        | llm  
        | StrOutputParser() 
    )
query = "등장인물에 대해 알려줘" # question
result = qa.invoke( query)
print( result )

등장인물로는 톰, 인준 조, 의사, 머프 포터, 헉, 미세스 더글라스, 그리고 교사가 있습니다. 톰은 사건의 목격자로 등장하며, 헉은 이야기를 듣고 있는 인물입니다. 미세스 더글라스는 헉에게 감사의 말을 전하는 인물입니다.
